# InterGATE V8 — Supplementary Figures S2 and S3

Figure-only notebook/script. It reads the locked summary tables produced by
`12_controles_revision_q_permutaciones.ipynb`; it does not retrain models or
recompute null distributions.

Outputs:

- `Supplementary_Figure_S2_factorial_q_control.{pdf,svg,png}`
- `Supplementary_Figure_S3_core_sensitivity.{pdf,svg,png}`


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/intergate-s2-s3-mpl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 1. Editable configuration

Leave both paths as `None` for automatic discovery. Set an explicit directory
only when the notebook is outside the InterGATE repository/results package.


In [ ]:
SOURCE_DIR_OVERRIDE: str | Path | None = None
OUTPUT_DIR_OVERRIDE: str | Path | None = None

PNG_DPI = 450
STRICT_VALIDATION = True
SHOW_FIGURES = True

S2_BASENAME = "Supplementary_Figure_S2_factorial_q_control"
S3_BASENAME = "Supplementary_Figure_S3_core_sensitivity"

S2_FIGSIZE = (14.2, 4.5)
S3_FIGSIZE = (12.8, 4.8)

STATISTICS = (
    "at_least_one_endpoint",
    "both_endpoints",
    "unique_node_fraction",
)
STATISTIC_LABELS = {
    "at_least_one_endpoint": "at least one endpoint",
    "both_endpoints": "both endpoints",
    "unique_node_fraction": "unique node fraction",
}

ARM_ORDER = (
    "InterGATE_raw",
    "InterGATE_module",
    "q_raw",
    "q_module",
)
ARM_TICK_LABELS = {
    "InterGATE_raw": "InterGATE\nraw",
    "InterGATE_module": "InterGATE\nmodule",
    "q_raw": "$q$\nraw",
    "q_module": "$q$\nmodule",
}
ARM_COLORS = {
    "InterGATE_raw": "#2F6FB0",
    "InterGATE_module": "#103D6E",
    "q_raw": "#E58B2A",
    "q_module": "#A94F00",
}

SELECTOR_COLORS = {
    "raw": "#4C78A8",
    "module": "#C44E52",
}


## 2. Locate and validate the locked source tables


In [ ]:
INPUT_FILES = (
    "factorial_directional_signature_tests.csv",
    "core_threshold_k_over_R.csv",
    "core_strict_subset_sensitivity.csv",
)


def _contains_all_inputs(path: Path) -> bool:
    return path.is_dir() and all((path / name).is_file() for name in INPUT_FILES)


def _candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if "__file__" in globals():
        roots.append(Path(__file__).resolve().parent)
    roots.append(Path.cwd().resolve())

    expanded: list[Path] = []
    for root in roots:
        expanded.extend([root, *root.parents])

    unique: list[Path] = []
    seen: set[Path] = set()
    for path in expanded:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


def locate_source_dir(override: str | Path | None = None) -> Path:
    candidates: list[Path] = []
    if override is not None:
        candidates.append(Path(override).expanduser().resolve())

    environment_path = os.environ.get("INTERGATE_REVIEWER_CONTROLS_DIR")
    if environment_path:
        candidates.append(Path(environment_path).expanduser().resolve())

    for root in _candidate_roots():
        candidates.extend([
            root / "source_data",
            root / "results_v8" / "12_reviewer_controls",
            root / "locked_results_v8" / "12_reviewer_controls",
            root / "12_reviewer_controls",
        ])

    checked: list[Path] = []
    seen: set[Path] = set()
    for path in candidates:
        path = path.resolve()
        if path in seen:
            continue
        checked.append(path)
        seen.add(path)
        if _contains_all_inputs(path):
            return path

    raise FileNotFoundError(
        "Could not locate the three locked source CSV files. Checked:\n- "
        + "\n- ".join(map(str, checked))
        + "\nSet SOURCE_DIR_OVERRIDE in the configuration cell."
    )


def locate_project_root() -> Path | None:
    for root in _candidate_roots():
        if (root / "intergate").is_dir() and (root / "notebooks").is_dir():
            return root
    return None


def locate_output_dir(override: str | Path | None = None) -> Path:
    if override is not None:
        output = Path(override).expanduser().resolve()
    else:
        project_root = locate_project_root()
        output = (
            project_root / "results_v8" / "13_supplementary_figures"
            if project_root is not None
            else Path.cwd().resolve() / "figures"
        )
    output.mkdir(parents=True, exist_ok=True)
    return output


SOURCE_DIR = locate_source_dir(SOURCE_DIR_OVERRIDE)
OUTPUT_DIR = locate_output_dir(OUTPUT_DIR_OVERRIDE)

factorial_signature = pd.read_csv(
    SOURCE_DIR / "factorial_directional_signature_tests.csv"
)
core_threshold_curve = pd.read_csv(
    SOURCE_DIR / "core_threshold_k_over_R.csv"
)
core_subset_sensitivity = pd.read_csv(
    SOURCE_DIR / "core_strict_subset_sensitivity.csv"
)

print(f"Source tables: {SOURCE_DIR}")
print(f"Figure output: {OUTPUT_DIR}")


In [ ]:
def validate_source_tables() -> None:
    factorial_required = {
        "arm", "statistic", "observed", "null_mean", "null_q025",
        "null_q975", "fold_over_null", "empirical_p_upper", "n_draws",
    }
    threshold_required = {
        "selector", "R", "k", "threshold", "n_edges", "n_nodes",
    }
    subset_required = {
        "selector", "subset_R", "seeds", "strict_edges_RofR",
        "strict_nodes_RofR",
    }
    for name, frame, required in (
        ("factorial", factorial_signature, factorial_required),
        ("threshold", core_threshold_curve, threshold_required),
        ("subsets", core_subset_sensitivity, subset_required),
    ):
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing columns {sorted(missing)}")
        if frame.empty:
            raise ValueError(f"{name}: empty input table")

    expected_pairs = {
        (arm, statistic) for arm in ARM_ORDER for statistic in STATISTICS
    }
    observed_pairs = set(zip(factorial_signature.arm, factorial_signature.statistic))
    if observed_pairs != expected_pairs or len(factorial_signature) != len(expected_pairs):
        raise ValueError("S2 requires exactly one row for every arm × statistic cell.")

    if STRICT_VALIDATION:
        if set(factorial_signature.n_draws.astype(int)) != {1000}:
            raise ValueError("S2 requires 1,000 candidate-matched null draws.")
        if set(core_threshold_curve.selector) != {"raw", "module"}:
            raise ValueError("S3 requires raw and module selectors.")
        r_values = set(core_threshold_curve.R.astype(int))
        if len(r_values) != 1:
            raise ValueError(f"S3 contains inconsistent R values: {r_values}")
        R = next(iter(r_values))
        for selector in ("raw", "module"):
            block = core_threshold_curve.loc[
                core_threshold_curve.selector.eq(selector)
            ].sort_values("k")
            if tuple(block.k.astype(int)) != tuple(range(1, R + 1)):
                raise ValueError(f"S3: incomplete k/R curve for {selector}.")
            counts = (
                core_subset_sensitivity.loc[
                    core_subset_sensitivity.selector.eq(selector)
                ].groupby("subset_R").size().to_dict()
            )
            expected_counts = {size: math.comb(R, size) for size in range(2, R + 1)}
            if counts != expected_counts:
                raise ValueError(
                    f"S3 seed-subset counts mismatch for {selector}: {counts}"
                )

validate_source_tables()
print("V8 source-table validation: PASS")


## 3. Common publication style and export helper


In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


def export_figure(fig: plt.Figure, basename: str) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    for extension in ("pdf", "svg", "png"):
        path = OUTPUT_DIR / f"{basename}.{extension}"
        options: dict[str, object] = {"bbox_inches": "tight"}
        if extension == "png":
            options["dpi"] = PNG_DPI
        fig.savefig(path, **options)
        paths[extension] = path
    return paths


## 4. Supplementary Figure S2

Four-cell factorial comparison of score source (InterGATE or \(q\)) and
selector (raw or module-aware), under the identical V8 Exact-K budget.


In [ ]:
def make_supplementary_figure_s2() -> tuple[plt.Figure, dict[str, Path]]:
    fig, axes = plt.subplots(
        1,
        3,
        figsize=S2_FIGSIZE,
        constrained_layout=True,
    )

    x = np.arange(len(ARM_ORDER))
    for axis, statistic in zip(axes, STATISTICS):
        block = (
            factorial_signature.loc[
                factorial_signature.statistic.eq(statistic)
            ].set_index("arm").loc[list(ARM_ORDER)]
        )
        values = block.fold_over_null.to_numpy(dtype=float)

        axis.bar(
            x,
            values,
            width=0.72,
            color=[ARM_COLORS[arm] for arm in ARM_ORDER],
        )
        axis.axhline(1.0, color="#555555", linestyle="--", linewidth=1)
        axis.set_xticks(x)
        axis.set_xticklabels([ARM_TICK_LABELS[arm] for arm in ARM_ORDER])
        axis.set_ylabel("Fold over candidate-matched null")
        axis.set_title(STATISTIC_LABELS[statistic])

        for x_position, value in enumerate(values):
            axis.text(
                x_position,
                value + 0.04,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    fig.suptitle(
        "Factorial control: learning × module-aware selection",
        fontsize=15,
        fontweight="bold",
    )
    paths = export_figure(fig, S2_BASENAME)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)
    return fig, paths


fig_s2, s2_paths = make_supplementary_figure_s2()
print("S2:", *s2_paths.values(), sep="\n  ")


## 5. Supplementary Figure S3

Panel A shows the number of edges retained at each \(k/5\) frequency
threshold. Panel B shows all strict intersections obtained from every subset
of \(R=2,3,4,5\) seeds.


In [ ]:
def make_supplementary_figure_s3() -> tuple[plt.Figure, dict[str, Path]]:
    fig, axes = plt.subplots(
        1,
        2,
        figsize=S3_FIGSIZE,
        constrained_layout=True,
    )

    # Panel A: complete k/5 threshold curves.
    for selector, block in core_threshold_curve.groupby("selector"):
        block = block.sort_values("k")
        axes[0].plot(
            block.k,
            block.n_edges,
            marker="o",
            linewidth=2.2,
            color=SELECTOR_COLORS[selector],
            label=selector,
        )
    axes[0].set_xticks(range(1, 6))
    axes[0].set_xlabel("Minimum selections k (R=5)")
    axes[0].set_ylabel("Edges retained at ≥k/R")
    axes[0].set_title("A  Threshold core curve")
    axes[0].legend(frameon=False)

    # Panel B: strict R/R intersections for every seed subset.
    positions = np.arange(2, 6)
    for offset, selector in zip((-0.13, 0.13), ("raw", "module")):
        blocks = [
            core_subset_sensitivity.loc[
                core_subset_sensitivity.selector.eq(selector)
                & core_subset_sensitivity.subset_R.eq(R),
                "strict_edges_RofR",
            ].to_numpy(dtype=float)
            for R in positions
        ]
        box = axes[1].boxplot(
            blocks,
            positions=positions + offset,
            widths=0.22,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
        )
        for patch in box["boxes"]:
            patch.set_facecolor(SELECTOR_COLORS[selector])
            patch.set_alpha(0.65)
        for median in box["medians"]:
            median.set_color("white")
            median.set_linewidth(1.7)

    axes[1].set_xticks(positions)
    axes[1].set_xlabel("Number of seeds in subset R")
    axes[1].set_ylabel("Strict intersection size R/R")
    axes[1].set_title("B  All seed-subset intersections")
    axes[1].plot(
        [], [], color=SELECTOR_COLORS["raw"], linewidth=7,
        alpha=0.65, label="raw",
    )
    axes[1].plot(
        [], [], color=SELECTOR_COLORS["module"], linewidth=7,
        alpha=0.65, label="module",
    )
    axes[1].legend(frameon=False)

    fig.suptitle(
        "Sensitivity of the Basal-like core definition",
        fontsize=15,
        fontweight="bold",
    )
    paths = export_figure(fig, S3_BASENAME)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)
    return fig, paths


fig_s3, s3_paths = make_supplementary_figure_s3()
print("S3:", *s3_paths.values(), sep="\n  ")


## 6. Final audit

PDF and SVG are vector outputs. PNG resolution is controlled by `PNG_DPI` in
the editable configuration cell.


In [ ]:
all_outputs = {**{f"S2_{k}": v for k, v in s2_paths.items()},
               **{f"S3_{k}": v for k, v in s3_paths.items()}}
missing_outputs = [str(path) for path in all_outputs.values() if not path.is_file()]
if missing_outputs:
    raise RuntimeError("Missing exported files: " + ", ".join(missing_outputs))

audit = pd.DataFrame({
    "artifact": list(all_outputs),
    "path": [str(path) for path in all_outputs.values()],
    "size_bytes": [path.stat().st_size for path in all_outputs.values()],
})
print(audit.to_string(index=False))
